<a href="https://colab.research.google.com/github/muraleee/collab-stuff/blob/main/notebooks/1.0-intro-langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -U langchain-core langchain-openai

In [ ]:
import os
import getpass

from google.colab import userdata



def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = userdata.get(var)

_set_env("OPENAI_API_KEY")
_set_env("ANTHROPIC_API_KEY")

In [ ]:
# option 1 with specific API provider Objects (ChatOpenAI, ChatAnthropic, ChatOllama....)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke("HI!")

In [ ]:
# RECOMMENDED: Use this method to initialize the LLM
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai")

model.invoke("Hi what is your model name?")

In [ ]:
# model = init_chat_model("claude-sonnet-4-5-20250929", model_provider="anthropic")

# model.invoke("Hi what is your name? Like the model name?")

# Full LLM App in LangChain

In [ ]:
import getpass
import os

try:
    # load environment variables from .env file (requires `python-dotenv`)
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

os.environ["LANGSMITH_TRACING"] = "true"
if "LANGSMITH_API_KEY" not in os.environ:
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
        prompt="Enter your LangSmith API key (optional): "
    )
if "LANGSMITH_PROJECT" not in os.environ:
    os.environ["LANGSMITH_PROJECT"] = getpass.getpass(
        prompt='Enter your LangSmith Project Name (default = "default"): '
    )
    if not os.environ.get("LANGSMITH_PROJECT"):
        os.environ["LANGSMITH_PROJECT"] = "default"
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        prompt="Enter your OpenAI API key (required if using OpenAI): "
    )

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-5-mini",
    model_provider="openai"
)

model.invoke("Hi")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage("Translate the following from English into Italian"),
    HumanMessage("Hi! Let's learn about large language models!"),
]

model.invoke(messages)

In [ ]:
model.invoke([{"role": "user", "content": "How are ya?"}])

In [ ]:
for token in model.stream("Tell me the 3 funniest jokes you know"):
    print(token.content, end="|")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from {language_source} into {language_target}"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("human", "{input_sentence}")]
)

prompt_template.format(language_source="English", language_target="French", input_sentence="I love programming in Python")

In [ ]:
prompt = prompt_template.invoke({"language_source": "English", "language_target": "French", "input_sentence": "I love programming in Python"})
response = model.invoke(prompt) # calling the model
response

In [ ]:
response.content

In [ ]:
chain = prompt_template | model

response = chain.invoke({"language_source": "English", "language_target": "Italian", "input_sentence": "Lucas is a gorgeous bald teacher."})

In [ ]:
response.content

# Recap

Langchain gives you components:
- model access
- prompt templates to create reusable strings with dynamic variables
- chains: blocks of re-usable computation with LLMs + prompt templates

# Structured Outputs

In [ ]:
from pydantic import BaseModel, Field
from openai import OpenAI

class ElementsOfLiveCourse(BaseModel):
    title: str = Field(description="The title of the live course")
    topic: str = Field(description="The core topic of the live course")
    example_lesson: str = Field(description="An example lesson from the live course")

with open("./course_example.md", "r") as f:
    prompt_raw_course = f.read()

client = OpenAI()

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You take in raw information for a course and\
            you output the structured objects with information about that course"},
        {"role": "user", "content": prompt_raw_course}
    ],
    response_format=ElementsOfLiveCourse
)

response

In [ ]:
!pip install -U --force-reinstall langchain-core
!pip install -U langchain langchain-openai langchain-community

In [ ]:
# Reinstall langchain-openai and langchain-core to ensure compatible versions
# !pip install --upgrade --force-reinstall langchain-openai langchain-core

from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# 1. Define your schema
class ElementsOfLiveCourse(BaseModel):
    title: str = Field(description="The title of the live course")
    topic: str = Field(description="The core topic of the live course")
    example_lesson: str = Field(description="An example lesson from the live course")

# 2. Load your data
# Ensure 'course_example.md' exists in your directory
with open("./course_example.md", "r") as f:
    prompt_raw_course = f.read()

# 3. Initialize the model
# Note: Ensure your OPENAI_API_KEY is set in your environment variables

llm = init_chat_model("gpt-5-mini", model_provider="openai")
# 4. Bind the structured output
structured_llm = llm.with_structured_output(ElementsOfLiveCourse)

# 5. Invoke and print
response = structured_llm.invoke(prompt_raw_course)

print(response)

In [ ]:
response.choices[0].message.parsed

In [ ]:
from IPython.display import Markdown

str_output = f"""
# {response.title}

- *Topic*: {response.topic}

### Example Lesson

{response.example_lesson}
"""
Markdown(str_output)

In [ ]:
llm = init_chat_model("gpt-5-mini", model_provider="openai")

llm.invoke("What are the elements of a Screenplay?")

## Pydantic Class

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field


# Pydantic
class ElementsOfScreenplay(BaseModel):
    """Elements of a Screenplay."""

    title: str = Field(description="The title of the screenplay")
    genre: str = Field(description="The genre of the screenplay")
    protagonist: str = Field(description="The protagonist of the screenplay")
    antagonist: str = Field(description="The antagonist of the screenplay")
    setting: str = Field(description="The setting of the screenplay")
    plot: str = Field(description="The plot of the screenplay")


structured_llm = llm.with_structured_output(ElementsOfScreenplay)

screenplay_structured = structured_llm.invoke("Structure a Screenplay about characters afraid of becoming outdated.")

In [ ]:
from IPython.display import Markdown

str_output = f"""
# {screenplay_structured.title}

- *Genre*: {screenplay_structured.genre}
- *Protagonist*: {screenplay_structured.protagonist}
- *Antagonist*
- *Setting*: {screenplay_structured.setting}

### Plot

{screenplay_structured.plot}
"""
Markdown(str_output)

# Model + Tools

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

model = init_chat_model("gpt-5-mini", model_provider="openai")
search = TavilySearchResults(max_results=2)
tools = [search]

model_with_tools = model.bind_tools(tools)

output_model_tools = model_with_tools.invoke("What is the latest model released by Anthropic?")

In [ ]:
output_model_tools.tool_calls

The output we get here is something called a "tool call" which means, prepared arguments for a pre-defined function (in this case
web search with the tavily API) to gather the required information.

Now let's look at a full agent.

In [ ]:
# Import relevant functionality
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

# Create the agent
memory = MemorySaver()
model = init_chat_model("gpt-4o-mini", model_provider="openai")
search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_react_agent(model, tools, checkpointer=memory)

agent_executor

Agent Executor is a graph!

In [ ]:
# We pass config because this agent has memory so we need to pass a thread_id
config = {"configurable": {"thread_id": "abc123"}}
# Below we can't just invoke on "input" we use "messages" because the agent expects a list of messages (this info is hidden in the create_react_agent function)
agent_executor.invoke({"messages": [HumanMessage("What is the capital of Brazil?")]}, config=config)

In [ ]:
for step in agent_executor.stream(
    {"messages": [HumanMessage(content="What are the best LLM models right now according to artificialanalysis.ai")]},
    stream_mode="values",
    config=config
):
    step["messages"][-1].pretty_print()

In [ ]:
# !pip install langchain-ollama

from langchain.chat_models import init_chat_model

local_llm = init_chat_model("mistral-small3.2", model_provider="ollama")

local_llm.invoke("What are some predictions for AI in 2026?")

In [ ]:
search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_react_agent(local_llm, tools)

In [ ]:
agent_executor.invoke({"messages": [HumanMessage("What are some predictions for AI in 2026?")]})